# 04 - 样本重叠检查

特别检查以下可能存在重复样本的队列对：
- GSE12251 与 GSE23597
- GSE14580 与 GSE16879

检查方法：
1. GSM 编号比较
2. 表达矩阵哈希比较
3. 表达矩阵相关性比较

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json
import hashlib
import numpy as np

base_dir = Path('..')
metadata_dir = base_dir / 'data' / 'metadata'

print('样本重叠检查')
print('='*60)

## 1. GSM 编号比较

In [ ]:
def load_gsm_list(accession):
    json_file = metadata_dir / f'{accession}_samples.json'
    if json_file.exists():
        with open(json_file) as f:
            meta = json.load(f)
        return [s['gsm'] for s in meta.get('samples', [])]
    soft_file = metadata_dir / f'{accession}_soft.txt'
    if soft_file.exists():
        gsms = []
        with open(soft_file, 'r', errors='replace') as f:
            for line in f:
                if line.startswith('^SAMPLE ='):
                    gsms.append(line.split('=')[1].strip())
        return gsms
    return []

# 检查 GSE12251 vs GSE23597
gsm_12251 = load_gsm_list('GSE12251')
gsm_23597 = load_gsm_list('GSE23597')
overlap_1 = set(gsm_12251) & set(gsm_23597)
print(f'GSE12251: {len(gsm_12251)} 样本')
print(f'GSE23597: {len(gsm_23597)} 样本')
print(f'GSM重叠: {len(overlap_1)} 个')
if overlap_1:
    print(f'重叠的GSM: {sorted(overlap_1)}')

# 检查 GSE14580 vs GSE16879
gsm_14580 = load_gsm_list('GSE14580')
gsm_16879 = load_gsm_list('GSE16879')
overlap_2 = set(gsm_14580) & set(gsm_16879)
print(f'\nGSE14580: {len(gsm_14580)} 样本')
print(f'GSE16879: {len(gsm_16879)} 样本')
print(f'GSM重叠: {len(overlap_2)} 个')
if overlap_2:
    print(f'重叠的GSM: {sorted(overlap_2)}')

## 2. 运行完整重叠检查脚本

In [ ]:
# 运行完整检查（包括表达矩阵哈希和相关性）
# !python ../scripts/check_sample_overlap.py

print('提示: 运行 check_sample_overlap.py 可执行完整检查')
print('包括: GSM编号比较 → 表达矩阵哈希 → 表达矩阵相关性')
print('结果保存到 results/sample_overlap_check.csv')

## 3. 重叠检查结果总结

In [ ]:
results_file = base_dir / 'results' / 'sample_overlap_check.csv'
if results_file.exists():
    results = pd.read_csv(results_file)
    print(results.to_string(index=False))
else:
    print('重叠检查结果文件不存在，需要先运行 check_sample_overlap.py')
    print()
    print('已知需要特别检查的队列对:')
    print('  1. GSE12251 vs GSE23597 (UC Infliximab)')
    print('  2. GSE14580 vs GSE16879 (UC Infliximab)')
    print()
    print('如果发现重叠样本，需要:')
    print('  - 从训练集中移除重复样本')
    print('  - 或在验证时排除重叠患者')
    print('  - 记录重叠情况以避免数据泄漏')